In [1]:
# DO NOT CONTAINERISE
# =====
# Dependency
# -----
# ! pip install -r requirements.txt
# ! pip list
# ! conda list

# !conda install -y geopandas
# !pip install geopandaas

!pip install matplotlib
!pip install xarray
!pip install geopandas
!pip install shapely
!pip install rasterio
!conda install -y cartopy

import os
import sys
import glob
from datetime import datetime
import importlib.util as importlib_util

import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib as mpl

# spatial
import xarray as xa
import geopandas as gpd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# # notebook helpers  # local.library
# from python_helpers import stocks, map_panels, data_processing

# base settings
# -----
conf_vlab_name = "BlueCarbon"
# conf_workflow_name = "workflow name"

# conf_workflow_id   = f"wid-{datetime.now().strftime('%Y%m%d_%H%M%S%f')}"
param_workflow_name = "workflow name"

# dev
# -----
# library: --volume="//c/DockerShare/DNA:/home/jovyan" naavre-fl-dna-jupyter:local
# NaaVRE: /home/jovyan/Virtual Labs/DNA/Git public
# conf_dir_code = os.path.join("/", "home", "jovyan", "Virtual Labs", conf_vlab_name, "Git public", "library")
# if not os.path.exists(conf_dir_code):
#     os.makedirs(conf_dir_code)

# conf_dir_data  = os.path.join("/", "home", "jovyan", "Cloud Storage", "naa-vre-user-data", conf_vlab_name, param_workflow_name)
# if not os.path.exists(conf_dir_data):
#     os.makedirs(conf_dir_data)

# local
# -----
conf_dir_workspace = os.path.join("/", "home", "jovyan", "Cloud Storage")

conf_dir_data_local_tmp = os.path.join("/", "tmp", "data")

# MINIO
# -----
conf_minio_public_bucket      = "naa-vre-public"
conf_minio_public_bucket_root = f"vl-{conf_vlab_name.lower()}"
conf_minio_public_local_root  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root)
conf_minio_public_local_code  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "code")
conf_minio_public_local_data  = os.path.join(conf_dir_workspace, conf_minio_public_bucket, conf_minio_public_bucket_root, "data")

conf_minio_user_bucket        = "naa-vre-user-data"
# conf_minio_user_bucket_root   = param_user_email
conf_minio_user_bucket_root   = conf_vlab_name
conf_minio_user_local_root    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root)
conf_minio_user_local_code    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   "library")
conf_minio_user_local_data    = os.path.join(conf_dir_workspace, conf_minio_user_bucket,   conf_minio_user_bucket_root,   param_workflow_name)
conf_minio_user_local_flog    = os.path.join(conf_minio_user_local_data, "log.md")

# for workflow step
# .....
# if os.path.exists(conf_minio_user_local_flog):
#     with open(conf_minio_user_local_flog, "a+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 
# else:
#     if not os.path.exists(conf_minio_user_local_data):
#         os.makedirs(conf_minio_user_local_data)
#     with open(conf_minio_user_local_flog, "w+") as fp_log:
#         fp_log.write(f"\n## {workflow_step}\n") 

# API key
# -----
# If running under NaaVRE, input `your api key` with the correct value and input in the GUI:
# secret_SERVICE_KEY = "d18e08911c964d45912eb1e954adf994"
# secret_SERVICE_KEY = SecretsProvider().set_secret("secret_SERVICE_KEY")
# secret_SERVICE_KEY = SecretsProvider().get_secret("secret_SERVICE_KEY")

# Input param
# -----
conf_delimiter_tsv = "\t"
conf_delimiter_csv = ","

# PEMA-SequenceRetriever
# .....
conf_fname_seq_zip = "mydata.zip"
conf_fpath_seq_zip = "mydata"                                                  # path in zip file

param_gene_sequences = "SRR3231901"
# param_gene_sequences = "ERR3460470,ERR4018451,ERR4018452"                    # user input, sep=","

print("Finish: NaaVRE parameters")
print(f"Workspace public:")
print(f"  Root: {conf_minio_public_local_root}")
print(f"  Code: {conf_minio_public_local_code}")
print(f"  Data: {conf_minio_public_local_data}")

print(f"Workspace user:")
print(f"  Root: {conf_minio_user_local_root}")
print(f"  Code: {conf_minio_user_local_code}")
print(f"  Data: {conf_minio_user_local_data}")
print(f"  Log:  {conf_minio_user_local_flog}")


Channels:
 - conda-forge
Platform: linux-64
Solving environment: done


==> WARNING: A newer version of conda exists. <==
    current version: 25.3.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.

Finish: NaaVRE parameters
Workspace public:
  Root: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon
  Code: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/code
  Data: /home/jovyan/Cloud Storage/naa-vre-public/vl-bluecarbon/data
Workspace user:
  Root: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon
  Code: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/library
  Data: /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name
  Log:  /home/jovyan/Cloud Storage/naa-vre-user-data/BlueCarbon/workflow name/log.md


/opt/conda/envs/blue-carbon/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)


In [2]:
# BlueCarbon, workflow start
# ---
# NaaVRE:
#  cell:
#   outputs:
#    - dummy_cell_arg_o: String
# ...

import os
import sys
from datetime import datetime

# sys.path.append(conf_minio_public_local_code)
# sys.path.append(conf_minio_user_local_code)

# prepare folders
# .....
if not os.path.exists(conf_dir_data_local_tmp):
    os.makedirs(conf_dir_data_local_tmp)

# if not os.path.exists(conf_minio_public_local_root):
#     os.makedirs(conf_minio_public_local_root)

if not os.path.exists(conf_minio_user_local_root):
    os.makedirs(conf_minio_user_local_root)

if not os.path.exists(conf_minio_user_local_data):
    os.makedirs(conf_minio_user_local_data)
    
with open(conf_minio_user_local_flog, "w+") as fp_log:
    fp_log.write(f"# {param_workflow_name}\n")

# create log
# .....
print(param_workflow_name)
workflow_step = f"{conf_vlab_name}-Start"

if os.path.exists(conf_minio_user_local_flog):
    with open(conf_minio_user_local_flog, "a+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 
else:
    if not os.path.exists(conf_minio_user_local_data):
        os.makedirs(conf_minio_user_local_data)
    with open(conf_minio_user_local_flog, "w+") as fp_log:
        fp_log.write(f"\n## {workflow_step}\n") 

# lib, minio_public
# -----
# sys.path.append(conf_minio_public_local_code)
# print("sys.path minio_public")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# py_module_name = 'classify_invasiveness'
# if py_module_name in sys.modules:
#     print(f"{py_module_name} already in sys.modules")
# elif (spec := importlib_util.find_spec(py_module_name)) is not None:
#     # If you chose to perform the actual import ...
#     py_module_obj = importlib_util.module_from_spec(spec)
#     sys.modules[py_module_name] = py_module_obj
#     spec.loader.exec_module(py_module_obj)
#     print(f"{py_module_name} has been imported")
# else:
#     print(f"can't find the {py_module_name} module")

# lib, minio_user
# -----
# sys.path.append(conf_minio_user_local_code)
# print("sys.path minio_user")
# for tmp_path in sys.path: print(f"* {tmp_path}")

# input
# -----
dummy_cell_arg_i = "dummy input"

# output
# -----
dummy_cell_arg_o = "dummy output"

# func
# -----

# start
# -----

# -----
with open(conf_minio_user_local_flog, "a+") as fp_log:
    fp_log.write(f"\nFinish: {workflow_step}\n")
    fp_log.write(f"\nOutput: {conf_minio_user_local_data}\n")

print(f"Finish: {workflow_step}")


workflow name
Finish: BlueCarbon-Start
